# Kaggle ローカル検証 — exp001 (gpt_oss)

Kaggle GPU notebook 上で `attack.py` を実モデル **gpt_oss**（gpt-oss-20b Q4）でリプレイ採点する。
Kaggle T4×2 は採点と同一 HW なので `public` は公開 LB と高相関、`provenance` は private 汎化の代理。
この notebook は採点には関与しない（提出用 attack notebook とは別物）。

## Kaggle 設定（右パネル）
- **Accelerator**: GPU T4 ×2（gpt-oss-20b Q4 ≈12GB は T4 16GB で快適）
- **Internet**: ON（pip の llama-cpp CUDA ビルドと GGUF DL に必須。電話番号認証が前提）
- **Add Input**: dataset `rikitomo0526/aiagent-security-validation` を追加
  （ローカルで `uv run python scripts/ops/build_validation_dataset.py` → `kaggle datasets create/version` でアップロード）

手順: ① 依存導入 → ② dataset 展開 → ③ GGUF 取得 → ④ smoke 実行 →（⑤ full 実行）。

## ① 依存（llama.cpp は CUDA ビルド）

In [ ]:
!pip -q install gymnasium 'pydantic>=2' huggingface_hub
import os
os.environ['CMAKE_ARGS'] = '-DGGML_CUDA=on'  # CUDA 版 llama-cpp-python
!pip -q install llama-cpp-python
import llama_cpp; print('llama_cpp OK')

## ② dataset を /kaggle/working へ展開（validation/ + vendor/ + attack.py）

Kaggle は dataset の `assets.zip` を自動展開する場合としない場合がある。両対応にする。

In [ ]:
import os, zipfile, shutil
SRC = '/kaggle/input/aiagent-security-validation'
WORK = '/kaggle/working'
zip_path = os.path.join(SRC, 'assets.zip')
if os.path.isfile(zip_path):
    # assets.zip がそのまま置かれている → working へ展開
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(WORK)
    print('assets.zip を展開しました')
else:
    # Kaggle が自動展開済み → input から working へコピー
    for d in ('validation', 'vendor'):
        shutil.copytree(os.path.join(SRC, d), os.path.join(WORK, d), dirs_exist_ok=True)
    shutil.copy(os.path.join(SRC, 'attack.py'), os.path.join(WORK, 'attack.py'))
    print('input から working へコピーしました')
os.chdir(WORK)
assert os.path.isdir('validation'), 'validation/ が無い: dataset を Add Input したか確認'
assert os.path.isdir('vendor/aicomp_sdk_pkg/aicomp_sdk'), 'SDK が無い: assets.zip を確認'
assert os.path.isfile('attack.py'), 'attack.py が無い'
print('展開 OK ->', sorted(os.listdir(WORK))[:8])

## ③ GGUF 取得（gpt_oss / 初回のみ・約12GB・Internet ON 必須）

In [ ]:
!python -m validation.download_models gpt_oss

## ④ smoke 実行（gpt_oss / 30 候補 / public,provenance / 約20分）

`public` = 公開 LB 相関、`provenance` = private 汎化代理。
期待: public で EXFIL+DEPUTY 発火、provenance で CONFUSED_DEPUTY のみ発火（述語×ガードレールの実証）。

In [ ]:
!python -m validation.run_validation \
    --attack /kaggle/working/attack.py --agent gpt_oss \
    --guardrails public,provenance \
    --candidates 30 --budget-s 600 --env gym \
    --summary-out /kaggle/working/summary_gpt_oss.json \
    --report-out /kaggle/working/report_gpt_oss.txt
print('--- report ---')
print(open('/kaggle/working/report_gpt_oss.txt').read())

## ⑤ full 実行（任意・夜間 8〜16h）と結果の持ち帰り

smoke が想定どおりなら本番候補数で回す（必要なら `strict` も足して `public,provenance,strict`）。
実行するには下のセルのコメントを外す。

実行後、Output の `summary_gpt_oss*.json` をローカルの `validation/runs/exp001/` に置き、
`/update-score exp001` で `docs/scores/SCORE.md` に反映する。`report` の述語内訳で EXFIL/DEPUTY の発火を確認。

In [ ]:
# !python -m validation.run_validation \
#     --attack /kaggle/working/attack.py --agent gpt_oss \
#     --guardrails public,provenance \
#     --candidates 620 --budget-s 8000 --env gym \
#     --summary-out /kaggle/working/summary_gpt_oss_full.json \
#     --report-out /kaggle/working/report_gpt_oss_full.txt